# 44 — Gemini Forensic Extension (replacement)

This replacement fixes the indentation error and preserves the fail-safe
behavior when `GEMINI_API_KEY` is missing.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

cfg = load_yaml(CONFIGS / "run.yml")

In [ ]:
step = "44_gemini_forensic_extension"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

api_key = (os.getenv("GEMINI_API_KEY", "") or "").strip()
model = (os.getenv("GEMINI_MODEL", "gemini-1.5-flash") or "gemini-1.5-flash").strip()

top, tmeta = safe_read_csv("outputs/43_window_ranking/window_top_candidates.csv")
if top.empty:
    raise FileNotFoundError("Run notebook 43 first.")

for c in ["window_start", "window_end"]:
    if c in top.columns:
        top[c] = pd.to_datetime(top[c], errors="coerce", utc=True).dt.tz_convert(None).dt.strftime("%Y-%m-%d")

lines = []
for _, r in top.head(10).iterrows():
    lines.append(
        f"- start={r.get('window_start')} end={r.get('window_end')} "
        f"days={r.get('window_days')} score={float(r.get('window_score', 0)):.3f} "
        f"abs_roll={float(r.get('abs_roll', 0)):.3f} "
        f"news_roll={float(r.get('news_roll', 0)):.3f} "
        f"ground_roll={float(r.get('ground_roll', 0)):.3f}"
    )
context = "\n".join(lines)

prompt = f'''You are assisting an environmental assessment project.
Use only the ranked candidate window metadata below.
Be cautious, neutral, and evidence-led.
Do not claim emissions or causation.
State where evidence is weak or indirect.

Return in JSON with keys:
overall_summary
priority_windows
recommended_follow_up
caveats

Candidate windows:
{context}
'''

write_json(out / "gemini_prompt.json", {"model": model, "prompt": prompt})

if not api_key:
    print("GEMINI_API_KEY missing; prompt saved only.")
    response_obj = {"status": "prompt_only", "reason": "missing GEMINI_API_KEY"}
    write_json(out / "gemini_response.json", response_obj)
else:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    r = requests.post(url, params={"key": api_key}, json=payload, timeout=90)
    try:
        resp = r.json()
    except Exception:
        resp = {"status_code": r.status_code, "text": r.text}
    write_json(out / "gemini_response.json", resp)

manifest = manifest_base(step, [CONFIGS / "run.yml"])
for p in [out / "gemini_prompt.json", out / "gemini_response.json"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
manifest["inputs"] = {"window_top_candidates": tmeta, "model": model, "api_key_present": bool(api_key)}
write_json(out / "manifest.json", manifest)
print("Wrote:", out)